# Resumable LLM Corpus Generator

Generate 15,000 short English LLM responses first; later set `TARGET_DOCS = 20000` to add 5,000. Saves checkpoints to Drive and resumes by document ID. **Do not delete the corpus file between sessions.**

Assignment: corpus average >=50 tokens. This notebook aims for concise, complete responses; it audits quality and writes progress logs. Generation speed is **not** a search benchmark.

In [ ]:
%pip -q install 'transformers>=4.44,<5' accelerate safetensors

In [ ]:
import os, json, time, random, statistics, hashlib
from pathlib import Path
from datetime import datetime, timezone
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

TARGET_DOCS = 15000  # After reaching 15K, change ONLY this to 20000 for the final 5K.
BATCH_SIZE = 8
CHECKPOINT_EVERY = 200
MAX_NEW_TOKENS = 128  # Safety ceiling, not target length
SEED = 20261019
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

from google.colab import drive
drive.mount("/content/drive")
ROOT = Path("/content/drive/MyDrive/HP-SS-Engine")
RAW = ROOT / "data" / "raw"
RAW.mkdir(parents=True, exist_ok=True)
CORPUS = RAW / "qwen_corpus_20k_progress.jsonl"
STATUS = RAW / "qwen_corpus_generation_status.json"
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU before large-scale generation.")


In [ ]:
TOPICS = [
"transformer attention","retrieval augmented generation","vector embeddings","model evaluation",
"prompt injection","phishing defense","network segmentation","password hashing",
"SQL indexing","database transactions","distributed caching","load balancing",
"API authentication","container security","cloud deployment","operating system scheduling",
"computer vision","image classification","speech recognition","machine translation",
"recommendation systems","anomaly detection","fraud detection","data privacy",
"electric vehicle charging","battery management","renewable energy","solar power",
"wind turbines","smart grids","carbon emissions","climate forecasting",
"earthquake monitoring","flood prediction","wildfire detection","water purification",
"public health analytics","vaccine logistics","medical imaging","hospital scheduling",
"financial risk","inflation trends","digital payments","supply chain forecasting",
"urban traffic","public transportation","road safety","satellite navigation",
"robot motion planning","autonomous vehicles","industrial automation","predictive maintenance",
"agricultural sensors","crop monitoring","food safety","ocean monitoring",
"education technology","adaptive learning","online misinformation","content moderation",
"document summarization","text classification","search ranking","semantic similarity",
"graph algorithms","time series forecasting","sensor fusion","edge computing",
"Internet of Things","wireless networks","data compression","software testing",
"version control","continuous integration","accessibility design","human computer interaction",
"energy efficient computing","parallel computing","GPU acceleration","memory bandwidth"
]
TASKS = [
"Explain the core idea of","Describe a practical application of","Compare two approaches to",
"Identify a common misconception about","Explain a reliability challenge in",
"Describe a simple example involving","Explain an ethical concern related to",
"Outline a beginner-friendly workflow for","Explain how to measure success in",
"Describe a failure mode of"
]
CONTEXTS = [
"for a small business","in a university lab","for a mobile application",
"in a public service","for a hospital team","in a rural community",
"for a startup","in a manufacturing plant","for an online platform",
"in an emergency response system","for a city government","for a research prototype",
"in a resource-constrained environment","for a customer support team",
"in a school setting","for a logistics company","for a security operations team",
"in a data center","for a transportation provider","in an environmental project",
"for a nonprofit organization","for a financial services team",
"in a consumer product","for an engineering student","in a remote monitoring system"
]
# 80 x 10 x 25 = 20,000 distinct topic-task-context combinations.
assert len(TOPICS)==80 and len(TASKS)==10 and len(CONTEXTS)==25
def prompt_for_id(doc_id):
    if not 0 <= doc_id < 20000:
        raise ValueError("doc_id outside 0..19999")
    # Reproducible pseudo-random permutation, no collisions across 20K IDs.
    idx = (doc_id * 7919 + 2713) % 20000
    topic = TOPICS[idx // 250]
    task = TASKS[(idx // 25) % 10]
    context = CONTEXTS[idx % 25]
    return f"{task} {topic} {context}."

assert len({prompt_for_id(i) for i in range(20000)})==20000
print("Unique prompts:",20000)
print("Examples:",*[prompt_for_id(i) for i in range(3)],sep="\n")


In [ ]:
t0=time.perf_counter()
tokenizer=AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left")
model=AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype="auto",device_map="auto").eval()
print(f"Model loaded in {time.perf_counter()-t0:.1f}s")
SYSTEM_MESSAGE = ("You write short factual English paragraphs for a semantic search dataset. "
"Write exactly 2 or 3 short, complete sentences totaling roughly 55 to 75 tokens. "
"Stay on topic. Never use lists, numbering, markdown, headings, or colons introducing lists. End with a complete sentence.")
def generate_batch(ids):
    prompts=[prompt_for_id(i) for i in ids]
    chats=[tokenizer.apply_chat_template(
        [{"role":"system","content":SYSTEM_MESSAGE},{"role":"user","content":p}],
        tokenize=False,add_generation_prompt=True) for p in prompts]
    enc=tokenizer(chats,return_tensors="pt",padding=True,truncation=True,max_length=384).to(model.device)
    with torch.inference_mode():
        outputs=model.generate(**enc,max_new_tokens=MAX_NEW_TOKENS,do_sample=False,
                               pad_token_id=tokenizer.eos_token_id)
    new_ids=outputs[:,enc.input_ids.shape[1]:]
    records=[]
    for i,p,token_ids in zip(ids,prompts,new_ids):
        text=tokenizer.decode(token_ids,skip_special_tokens=True).strip()
        count=len(tokenizer.encode(text,add_special_tokens=False))
        records.append({"id":i,"prompt":p,"response":text,"response_tokens":count,
                        "model":MODEL_NAME})
    return records

# Quick 100-document quality pilot; do not add these records to the final corpus yet.
pilot_start=time.perf_counter()
pilot=[]
for j in range(0,100,BATCH_SIZE):
    pilot.extend(generate_batch(list(range(j,min(j+BATCH_SIZE,100)))))
elapsed=time.perf_counter()-pilot_start
lens=[x["response_tokens"] for x in pilot]
cap=sum(x["response_tokens"]>=MAX_NEW_TOKENS-1 for x in pilot)
print(f"100-doc pilot: {elapsed:.1f}s; {100/elapsed:.2f} docs/s")
print(f"Mean tokens: {statistics.mean(lens):.1f}; min: {min(lens)}; max: {max(lens)}")
print(f"At token cap: {cap}/100; empty: {sum(not x['response'] for x in pilot)}/100")
print("Sample 1:",pilot[0]["response"])
print("Sample 2:",pilot[1]["response"])
pilot_path=RAW/"qwen_quality_pilot_100.jsonl"
with pilot_path.open("w",encoding="utf-8") as f:
    for row in pilot: f.write(json.dumps(row,ensure_ascii=False)+"\n")
if statistics.mean(lens)<50:
    raise RuntimeError("Mean token count is below assignment minimum; adjust prompt before large run.")
if cap>10:
    raise RuntimeError("Too many pilot responses hit token ceiling; revise generation before production.")


In [ ]:
def read_existing(path):
    if not path.exists():
        return {}
    rows={}
    bad=0
    with path.open("r",encoding="utf-8") as f:
        for line in f:
            if not line.strip(): continue
            try:
                item=json.loads(line)
                i=item["id"]
                if not isinstance(i,int) or not (0<=i<20000): raise ValueError("invalid id")
                if i in rows: raise ValueError("duplicate id")
                if item["prompt"]!=prompt_for_id(i): raise ValueError("prompt mismatch")
                rows[i]=item
            except (json.JSONDecodeError,ValueError,KeyError,TypeError):
                bad+=1
    if bad:
        raise RuntimeError(f"Corpus has {bad} invalid/duplicate lines. Fix file before resuming.")
    return rows

existing=read_existing(CORPUS)
print(f"Found {len(existing)} existing documents. Target: {TARGET_DOCS}.")
assert TARGET_DOCS in (15000,20000)
remaining=[i for i in range(TARGET_DOCS) if i not in existing]
print(f"Remaining: {len(remaining)}. File: {CORPUS}")


In [ ]:
# RUN THIS CELL to generate/resume the full corpus.
# If Colab disconnects, reconnect, rerun earlier setup cells and rerun this cell.
# Keep TARGET_DOCS=15000 until phase one is finished.
start=time.perf_counter()
completed=0
recent=[]
with CORPUS.open("a",encoding="utf-8") as f:
    for j in range(0,len(remaining),BATCH_SIZE):
        ids=remaining[j:j+BATCH_SIZE]
        records=generate_batch(ids)
        for row in records:
            f.write(json.dumps(row,ensure_ascii=False)+"\n")
        completed+=len(records)
        recent.extend(records)
        if completed % CHECKPOINT_EVERY==0 or j+BATCH_SIZE>=len(remaining):
            f.flush()
            os.fsync(f.fileno())
            elapsed=time.perf_counter()-start
            stats={"timestamp_utc":datetime.now(timezone.utc).isoformat(),
                   "target_docs":TARGET_DOCS,"previous_docs":len(existing),
                   "new_docs_this_session":completed,
                   "total_docs":len(existing)+completed,
                   "elapsed_seconds_this_session":round(elapsed,2),
                   "model":MODEL_NAME,"batch_size":BATCH_SIZE,
                   "mean_new_tokens_this_session":round(statistics.mean(x["response_tokens"] for x in recent),2)}
            STATUS.write_text(json.dumps(stats,indent=2),encoding="utf-8")
            print(f"Checkpoint: {stats['total_docs']}/{TARGET_DOCS} docs, "
                  f"{elapsed/60:.1f} min this session, "
                  f"{completed/max(elapsed,0.01):.2f} docs/s",flush=True)
print("Session complete. Final file:",CORPUS)


In [ ]:
# Audit the accumulated corpus after each session.
rows=read_existing(CORPUS)
lengths=[r["response_tokens"] for r in rows.values()]
print("Count:",len(rows),"unique IDs:",len(set(rows)))
if lengths:
    print("Mean tokens:",round(statistics.mean(lengths),2))
    print("Min / max tokens:",min(lengths),max(lengths))
    print("Empty responses:",sum(not r["response"] for r in rows.values()))
    print("At cap:",sum(n>=MAX_NEW_TOKENS-1 for n in lengths))
    print("Exact duplicate response texts:",len(rows)-len({r["response"] for r in rows.values()}))
    print("First response:",rows[min(rows)]["response"])
    assert statistics.mean(lengths)>=50, "Dataset average below requirement"
print("Complete to target:",len(rows)>=TARGET_DOCS)
